## DECK & CARDS CLASSES

In [ ]:
import random
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

class Deck:
    def __init__(self):
        # initialize the deck with 92 cards, 8 cards of each number for each color, 1 card of 0 for each color, 4 cards of Skip for each color
        self.cards = []
        for color in ['Red', 'Green', 'Blue', 'Yellow']:
            for number in range(1,10):
                for _ in range(2):    # 2 copies per color
                    strNum = str(number)
                    self.cards.append(Card(color,strNum))
            self.cards.append(Card(color,'0'))    # 1 0 for each color
            for i in range(4):
                self.cards.append(Card(color, "Skip")) # 4 cards of Skip for each color
        # shuffle the deck
        random.shuffle(self.cards)

    def draw_card(self):
        return self.cards.pop()


## NECESSARY GAME FUNCTIONS

In [ ]:
def valid_move(card, top_card):
    if(card.value == top_card.value):
        return True
    if(card.color == top_card.color):
        return True
    return False

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        x = valid_move(card, top_card)
        if(x == True):
            valid_moves.append(card)
    return valid_moves


def deal_cards(deck, num_players=3, cards_per_player=5):
    hands = []
    for i in range(num_players):
        hand = []  # this for each player, end par we will append to hands list
        for j in range(cards_per_player):
            hand.append(deck.draw_card())
        hands.append(hand)
    return hands

def initial_state():
    deck = Deck()                           # creates and shuffles 92 cards
    hands = deal_cards(deck)               # deal 5 cards to each player
    top_card = deck.cards.pop()            # flip top card to start

    # make sure top card is not a skip
    while(top_card.value == "Skip"):
        deck.cards.append(top_card)        # put skip back
        random.shuffle(deck.cards)
        top_card = deck.cards.pop()

    state = {
        'hands': hands,                    # [[p1 cards], [p2 cards], [p3 cards]]
        'top_card': top_card,
        'deck': deck.cards,                
        'current_player': 0,              # 0=player#1, 1=player#2, 2=player#3
            }
    return state

def apply_move(state, move):
    new_state = copy.deepcopy(state)    # we do this to ensure that old state not changed
    current = new_state['current_player']
    hand = new_state['hands'][current]  # get current player's hand
    top_card = new_state['top_card']
    deck = new_state['deck']
    valid_moves = get_valid_moves(hand,top_card)
    if(len(valid_moves) == 0): 
        hand.append(deck.pop())
        next_player = (current + 1) % 3
    else:
        hand.remove(move)
        new_state['top_card'] = move
        if(move.value == "Skip"):
            new_player = (current + 2) % 3
        else:
            new_player = (current + 1) % 3
    new_state['current_player'] = new_player
    return new_state


def evaluation(state, player_index, strategy, num_players=3):
    player_hand = state['hands'][player_index]
    COPP = 0
    CAI = len(player_hand)
    S = 0
    for opponent in range(num_players):
        if(opponent == player_index):   # skip player index
            continue 
        opponent_hand = state['hands'][opponent]
        COPP += len(opponent_hand)
    for card in player_hand:
        if(card.value == "Skip"):
            S += 1

    COPP = COPP / (num_players - 1)   # since COPP is average num cards in oppoenets so num_players - 1
    if (strategy == "defensive"):
        return 50 - 7*CAI + 2*COPP + 4*S
    elif(strategy == "offensive"):                                  
        return 50 - 5*CAI + 3*COPP + 2*S